In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
target = "is_fraud"

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666085 entries, 0 to 666084
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   timestamp            666085 non-null  object 
 1   transaction_id       666085 non-null  object 
 2   account_id           666085 non-null  object 
 3   hour_of_day          666085 non-null  int64  
 4   day_of_week          666085 non-null  int64  
 5   is_weekend           666085 non-null  int64  
 6   amount               666085 non-null  float64
 7   merchant_category    666085 non-null  object 
 8   mcc_code             666085 non-null  int64  
 9   merchant_country     666085 non-null  object 
 10  card_present         666085 non-null  int64  
 11  device_type          666085 non-null  object 
 12  device_known         666085 non-null  int64  
 13  ip_risk_score        666085 non-null  float64
 14  is_foreign_txn       666085 non-null  int64  
 15  time_since_last_s

In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166413 entries, 0 to 166412
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   timestamp            166413 non-null  object 
 1   transaction_id       166413 non-null  object 
 2   account_id           166413 non-null  object 
 3   hour_of_day          166413 non-null  int64  
 4   day_of_week          166413 non-null  int64  
 5   is_weekend           166413 non-null  int64  
 6   amount               166413 non-null  float64
 7   merchant_category    166413 non-null  object 
 8   mcc_code             166413 non-null  int64  
 9   merchant_country     166413 non-null  object 
 10  card_present         166413 non-null  int64  
 11  device_type          166413 non-null  object 
 12  device_known         166413 non-null  int64  
 13  ip_risk_score        166413 non-null  float64
 14  is_foreign_txn       166413 non-null  int64  
 15  time_since_last_s

In [ ]:
train.head()

In [ ]:
test.head()

In [121]:
train = train.drop(columns=['fraud_pattern', 'transaction_id'], errors='ignore')
test = test.drop(columns=['fraud_pattern', 'transaction_id'], errors='ignore')

In [122]:
train['timestamp'] = pd.to_datetime(train['timestamp'])
train.sort_values(by='timestamp', inplace = True)
train.set_index('timestamp', inplace=True)

In [123]:
test['timestamp'] = pd.to_datetime(test['timestamp'])
test.sort_values(by='timestamp', inplace = True)
test.set_index('timestamp', inplace=True)

In [124]:
num_cols = ['hour_of_day', 'day_of_week', 'is_weekend', 'amount', 'mcc_code', 'card_present', 'device_known', 'ip_risk_score', 'is_foreign_txn', 'time_since_last_s', 'velocity_1h', 'amount_vs_avg_ratio', 'account_age_days', 'has_2fa', 'credit_limit', 'is_fraud']
cate_cols = ['account_id', 'merchant_category', 'merchant_country', 'device_type']

In [125]:
#Safe Division and Safe Logarithm functions
def safe_divide(num, denom, fill=0):
    return (num / (denom.replace(0, np.nan) + 1e-10)).replace([np.inf,-np.inf], fill).fillna(fill)

def safe_log(s, fill=0):
    return pd.Series(np.log1p(np.maximum(s, 0)), index=s.index).replace([np.inf,-np.inf], fill).fillna(fill)

In [126]:
#Amount features 
def add_amount_features(df):
    df = df.copy()
    
    # Find amount column
    amt_col = next((c for c in ['amount'] if c in df.columns), None)
    if amt_col is None:
        raise ValueError("No 'amount' column found in DataFrame")
    
    # Feature engineering
    df['log_amount']       = safe_log(df[amt_col])
    df['is_round_amount']  = ((df[amt_col] % 10 == 0) & (df[amt_col] > 0)).astype(int)
    df['high_amount_flag'] = (df[amt_col] > df[amt_col].quantile(0.95)).astype(int)
    df['low_amount_flag']  = (df[amt_col] < df[amt_col].quantile(0.05)).astype(int)
    
    new_var = ['log_amount', 'is_round_amount', 'high_amount_flag', 'low_amount_flag']
    
    print("Amount features added")
    
    return df, new_var

In [127]:
def add_time_features(df, new_var):
    df = df.copy()
    
    # Force a datetime column (fallback to index if needed)
    time_col = next((c for c in ['datetime','date','transaction_date','created_at'] if c in df.columns), None)

    dt = pd.to_datetime(
        df[time_col] if time_col else df.index,
        errors='coerce'
    )

    # Feature engineering
    df['is_night'] = df['hour_of_day'].between(0, 5).astype(int)
    df['is_business_hours'] = df['hour_of_day'].between(9, 17).astype(int)
    df['is_late_night'] = df['hour_of_day'].between(0, 3).astype(int)

    # Cycling features
    h = df["hour_of_day"].astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * h / 24)
    df["hour_cos"] = np.cos(2 * np.pi * h / 24)
    df = df.drop(columns=["hour_of_day"], errors="ignore")

    new_var += ['is_night', 'is_business_hours', 'is_late_night', 'hour_sin', 'hour_cos']

    print('Time features added')
    
    return df, new_var

In [128]:
def add_risk_features(df, new_var):
    df = df.copy()
    
    #Risk features & Utilization
    # 1) IP risk score features
    if "ip_risk_score" in df.columns:
        df["log_ip_risk_score"] = safe_log(df["ip_risk_score"])
        df["high_ip_risk_flag"] = (df["ip_risk_score"] > df["ip_risk_score"].quantile(0.80)).astype(int)
        new_var += ["log_ip_risk_score", "high_ip_risk_flag"]

    # 2) Amount vs average ratio (spike)
    if "amount_vs_avg_ratio" in df.columns:
        df["log_amount_vs_avg_ratio"] = safe_log(df["amount_vs_avg_ratio"])
        df["high_amount_vs_avg_flag"] = (df["amount_vs_avg_ratio"] > df["amount_vs_avg_ratio"].quantile(0.90)).astype(int)
        new_var += ["log_amount_vs_avg_ratio", "high_amount_vs_avg_flag"]

    # 3) Velocity features
    if "velocity_1h" in df.columns:
        df["log_velocity_1h"] = safe_log(df["velocity_1h"])
        df["high_velocity_1h_flag"] = (df["velocity_1h"] > df["velocity_1h"].quantile(0.90)).astype(int)
        new_var += ["log_velocity_1h", "high_velocity_1h_flag"]

    # 4) Time since last transaction (short gap = suspicious)
    if "time_since_last_s" in df.columns:
        df["log_time_since_last_s"] = safe_log(df["time_since_last_s"])
        df["low_time_since_last_flag"] = (df["time_since_last_s"] < df["time_since_last_s"].quantile(0.10)).astype(int)
        new_var += ["log_time_since_last_s", "low_time_since_last_flag"]

    # 5) Utilization (amount / credit_limit)
    amt_col = "amount" if "amount" in df.columns else None
    if ("credit_limit" in df.columns) and (amt_col is not None):
        df["utilization"] = safe_divide(df[amt_col], df["credit_limit"])
        df["high_utilization_flag"] = (df["utilization"] > df["utilization"].quantile(0.90)).astype(int)
        new_var += ["utilization", "high_utilization_flag"]

    return df, new_var

In [129]:
target = 'is_fraud'

**Logistic Regression Encode**

In [139]:
#Onehot Encode for categorical variables
def onehot_encode(df, target='is_fraud'):
    import pandas as pd
    
    df = df.copy()
    
    cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    cat_cols = [c for c in cat_cols if c != target]

    #remove account_id if exists
    if 'account_id' in cat_cols:
        cat_cols.remove('account_id')
        df = df.drop(columns=['account_id'], errors='ignore')
        print("Removed 'account_id' from features")
    
    # fill missing
    df[cat_cols] = df[cat_cols].fillna('MISSING')
    
    # one-hot
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    
    print(f"✅ One-hot encoded {len(cat_cols)} categorical columns")
    
    return df_encoded, cat_cols

In [140]:
#Scale numeric variables
def scale_numeric(df, target='is_fraud'):
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    
    df = df.copy()
    
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != target]
    
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    
    print(f"✅ Scaled {len(num_cols)} numeric columns")
    
    return df, scaler, num_cols

In [141]:
# =========================
# TRAIN PROCESSING
# =========================
new_var = []
train_log = train.copy()

# Feature engineering
train_log, new_var = add_amount_features(train_log)
train_log, new_var = add_time_features(train_log, new_var)
train_log, new_var = add_risk_features(train_log, new_var)

# One-hot encoding
train_log, cat_cols = onehot_encode(train_log, target)

# Scaling (FIT here)
train_log, scaler, num_cols = scale_numeric(train_log, target)

# Save train columns (VERY IMPORTANT)
train_columns = train_log.columns

# Save processed train
train_log.to_csv("train_log_processed.csv", index=False)


# =========================
# TEST PROCESSING
# =========================
new_var_test = []
test_log = test.copy()

# Feature engineering (same)
test_log, new_var_test = add_amount_features(test_log)
test_log, new_var_test = add_time_features(test_log, new_var_test)
test_log, new_var_test = add_risk_features(test_log, new_var_test)

# One-hot encoding
test_log, _ = onehot_encode(test_log, target)

# Align columns with train
test_log = test_log.reindex(columns=train_columns, fill_value=0)

# Scaling (IMPORTANT: only transform)
test_log[num_cols] = scaler.transform(test_log[num_cols])

# Save processed test
test_log.to_csv("test_log_processed.csv", index=False)

Amount features added
Time features added
Removed 'account_id' from features
✅ One-hot encoded 3 categorical columns
✅ Scaled 33 numeric columns
Amount features added
Time features added
Removed 'account_id' from features
✅ One-hot encoded 3 categorical columns


In [147]:
train_log.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 666085 entries, 2022-01-01 00:00:01 to 2023-12-31 23:59:42
Data columns (total 61 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   day_of_week                       666085 non-null  float64
 1   is_weekend                        666085 non-null  float64
 2   amount                            666085 non-null  float64
 3   mcc_code                          666085 non-null  float64
 4   card_present                      666085 non-null  float64
 5   device_known                      666085 non-null  float64
 6   ip_risk_score                     666085 non-null  float64
 7   is_foreign_txn                    666085 non-null  float64
 8   time_since_last_s                 666085 non-null  float64
 9   velocity_1h                       666085 non-null  float64
 10  amount_vs_avg_ratio               666085 non-null  float64
 11  account_age_days  

In [ ]:
#train_log.to_parquet(".../train_log_processed.parquet", index=False)

In [ ]:
#test_log.to_parquet(".../test_log_processed.parquet", index=False)

**Tree Models Encode**

In [130]:
def label_encode(df, target='is_fraud'):
    from sklearn.preprocessing import LabelEncoder
    import numpy as np
    
    df = df.copy()
    encoders = {}
    
    for col in cate_cols:
        if col == target:
            continue
            
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        
        print(f"  ✅ Encoded '{col}' ({df[col].nunique()} cats)")
    
    # ===== Handle inf =====
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # ===== Fill NA numeric =====
    num_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in num_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
    
    return df, encoders

In [131]:
# =========================
# TRAIN PROCESSING
# =========================
new_var = []
train_tree = train.copy()

# Feature engineering
train_tree, new_var = add_amount_features(train_tree)
train_tree, new_var = add_time_features(train_tree, new_var)
train_tree, new_var = add_risk_features(train_tree, new_var)

# Label-encoding
train_tree, encoders = label_encode(train_tree, target)

# Save train columns (VERY IMPORTANT)
train_columns = train_tree.columns

# Save processed train
train_tree.to_csv("train_tree_processed.csv", index=False)


# TEST PROCESSING
test_tree = test.copy()

test_tree, _ = add_amount_features(test_tree)
test_tree, _ = add_time_features(test_tree, [])
test_tree, _ = add_risk_features(test_tree, [])

# ✅ đúng
test_tree, _ = label_encode(test_tree, target=target)

# align columns
test_tree = test_tree.reindex(columns=train_columns, fill_value=0)

# save
test_tree.to_csv("test_tree_processed.csv", index=False)

Amount features added
Time features added
  ✅ Encoded 'account_id' (49633 cats)
  ✅ Encoded 'merchant_category' (14 cats)
  ✅ Encoded 'merchant_country' (11 cats)
  ✅ Encoded 'device_type' (5 cats)
Amount features added
Time features added
  ✅ Encoded 'account_id' (43652 cats)
  ✅ Encoded 'merchant_category' (14 cats)
  ✅ Encoded 'merchant_country' (11 cats)
  ✅ Encoded 'device_type' (5 cats)


In [135]:
test_tree.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 166413 entries, 2024-01-01 00:01:02 to 2024-06-30 23:59:53
Data columns (total 38 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   account_id                166413 non-null  int64  
 1   day_of_week               166413 non-null  int64  
 2   is_weekend                166413 non-null  int64  
 3   amount                    166413 non-null  float64
 4   merchant_category         166413 non-null  int64  
 5   mcc_code                  166413 non-null  int64  
 6   merchant_country          166413 non-null  int64  
 7   card_present              166413 non-null  int64  
 8   device_type               166413 non-null  int64  
 9   device_known              166413 non-null  int64  
 10  ip_risk_score             166413 non-null  float64
 11  is_foreign_txn            166413 non-null  int64  
 12  time_since_last_s         166413 non-null  int64  
 13  velocity_1

In [ ]:
#train_tree.to_parquet(".../train_tree_processed.parquet", index=False)

In [ ]:
#test_tree.to_parquet(".../test_tree_processed.parquet", index=False)